# Reproduce Tables 7–10 and the grid-resolution controls from the saved scores

**Paper:** *The ROC–PR Divergence and Threshold Transferability: A Cost-Calibrated Evaluation Protocol for Fraud Detection* (International Journal of Data Science and Analytics, revision).

This notebook **retrains nothing**. It reads the raw validation and test scores of the six D1 (creditcard) model configurations at seed 42, written by the main pipeline (`c3e_framework.prepare_splits` + the trainers of `c3e_framework` / `c3e_contribution3`), and regenerates from them:

| Section | Paper item | Output |
|---|---|---|
| 1 | saved scores, Temperature-Scaling and Beta maps | `results/scores/scores_d1_seed42_calibrated.npz`, `.csv` |
| 2 | Table 7 (extended comparison on D1: Δ, ECE, cost, flagged count per calibrator) | `results/tab7_extended_d1.csv` |
| 3 | Table 8 (three grids × models; τ̂*, validation cost, ties, flagged, test cost, oracle; null-model cost 750) | `results/tab8_gridres.csv` |
| 4 | ε sweep 10⁻⁶ … 10⁻¹⁵ and the rank-map control (Section 6.2) | `results/tab_eps_sensitivity.csv` |
| 5 | Δ on raw vs mapped scores (Section 6.2, Proposition 3.1) | `results/tab_delta_after_calibration.csv` |
| 6 | Remark 3.3: Reviewer-2 counter-example and the 2,000-draw simulation (seed 42) | `results/tab_counterexample.csv` |
| 7 | Table 10: the 26 pairs with Δ, cost, costly/regular label, and the κ sweep | `results/tab10_pairs_labelled.csv`, `results/tab10_screening.csv` |
| 8 | Table 9: fixed vs logarithmic cost regime (and the linear regime of Fig. 11) | `results/tab9_costregimes.csv`, `results/fig11_cost_regimes_d1.csv` |
| 9 | Assertions: every number quoted in the paper is checked against the regenerated values | printed report |

Dependencies: `numpy`, `pandas`, `scipy`, `scikit-learn` only. The two post-hoc maps are copied verbatim from `c3e_contribution3.py` (same `eps = 1e-7` clip, same optimiser) so that the notebook runs without the boosting libraries; if `c3e_contribution3` is importable, Section 1 checks that both implementations agree.

Table 10 needs the five-seed results of the six-dataset benchmark (`results/c3e_all_seeds_raw.csv`, written by Section 2b of `C3E_master_notebook.ipynb`).

In [1]:
from pathlib import Path
import json, warnings
import numpy as np, pandas as pd
from scipy.special import expit
from scipy.optimize import minimize_scalar, minimize
from sklearn.metrics import roc_auc_score, average_precision_score
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

RES = Path("results"); SCORES = RES / "scores"
SEED = 42                       # reference seed of every D1 result and of the simulation of Section 6
CFN, CFP = 10.0, 1.0            # fixed-cost regime
TAU_BAYES = CFP / (CFP + CFN)
N_GRID = 1001                   # uniform grid of the protocol
DELTA_STAR_LIST = [0.10, 0.15, 0.20, 0.25, 0.30]
KAPPA_LIST, KAPPA_MAIN = [1.5, 2.0, 5.0], 2.0
EPS_LIST = [1e-6, 1e-7, 1e-9, 1e-12, 1e-15]     # 1e-7 is the value of the released code
MODELS = ["LR", "RF", "XGB", "LGBM"]             # core models (Tables 8 and 10)
SIX = ["LR", "RF", "XGB", "LGBM", "CatBoost", "IF-Hybrid"]

Z = np.load(SCORES / f"scores_d1_seed{SEED}.npz")
y_val, y_te = Z["y_val"].astype(int), Z["y_te"].astype(int)
amount_val, amount_te = Z["amount_val"].astype(float), Z["amount_te"].astype(float)
scores = {m: dict(val=Z[f"{m}_val"].astype(float), test=Z[f"{m}_test"].astype(float)) for m in SIX}
NULL_COST_VAL, NULL_COST_TEST = CFN * int(y_val.sum()), CFN * int(y_te.sum())
print(f"N_val={len(y_val):,} ({int(y_val.sum())} frauds)   N_test={len(y_te):,} ({int(y_te.sum())} frauds)")
print(f"null-model cost: validation {NULL_COST_VAL:.0f} | test {NULL_COST_TEST:.0f}")
for m in SIX:
    print(f"  {m:10s} distinct validation scores: {np.unique(scores[m]['val']).size:6d}")

N_val=56,961 (57 frauds)   N_test=56,962 (75 frauds)
null-model cost: validation 570 | test 750
  LR         distinct validation scores:  55855
  RF         distinct validation scores:    117
  XGB        distinct validation scores:  55089
  LGBM       distinct validation scores:     21
  CatBoost   distinct validation scores:  36293
  IF-Hybrid  distinct validation scores:  55115


## 1 — Post-hoc maps (verbatim copies of `c3e_contribution3.py`) and calibrated scores

Both maps clip their input to $[\varepsilon, 1-\varepsilon]$ with $\varepsilon = 10^{-7}$ before taking logarithms; as implemented they are therefore non-decreasing but not injective (Section 6.2 of the paper).

In [2]:
class TemperatureScaling:
    def __init__(self, eps=1e-7): self.eps, self.T_ = eps, 1.0
    def _logit(self, s):
        e = self.eps
        return np.log(np.clip(s, e, 1 - e) / np.clip(1 - s, e, 1 - e))
    def fit(self, s, y):
        lg, e = self._logit(np.asarray(s, float)), self.eps
        def nll(logT):
            p = np.clip(expit(lg / np.exp(logT)), e, 1 - e)
            return -float((y * np.log(p) + (1 - y) * np.log(1 - p)).mean())
        self.T_ = float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x)); return self
    def predict(self, s): return expit(self._logit(np.asarray(s, float)) / self.T_)

class BetaCalibration:
    def __init__(self, eps=1e-7): self.eps, self.params_ = eps, np.array([1.0, 1.0, 0.0])
    def fit(self, s, y):
        e = self.eps; p = np.clip(np.asarray(s, float), e, 1 - e)
        def nll(params):
            a, b, c = params
            pc = np.clip(expit(a * np.log(p) - b * np.log(1 - p) + c), e, 1 - e)
            return -float((y * np.log(pc) + (1 - y) * np.log(1 - pc)).mean())
        self.params_ = minimize(nll, x0=self.params_, method="L-BFGS-B",
                                bounds=[(1e-4, None), (1e-4, None), (None, None)]).x
        return self
    def predict(self, s):
        e = self.eps; p = np.clip(np.asarray(s, float), e, 1 - e); a, b, c = self.params_
        return expit(a * np.log(p) - b * np.log(1 - p) + c)

def ece_10bins(y, p):
    # identical to c3e_contribution3.calibration_metrics (10 equal-width bins on [0, 1))
    bins, ece = np.linspace(0, 1, 11), 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (p >= lo) & (p < hi)
        if mask.sum() == 0: continue
        ece += mask.sum() / len(y) * abs(float(y[mask].mean()) - float(p[mask].mean()))
    return round(ece, 4)

def fit_map(kind, p_val, y):
    if kind == "none":        return lambda p: np.asarray(p, float)
    if kind == "temperature": return TemperatureScaling().fit(p_val, y).predict
    if kind == "beta":        return BetaCalibration().fit(p_val, y).predict
    raise ValueError(kind)

# calibrated scores for every model, written next to the raw ones
CAL = ["none", "temperature", "beta"]
cal_scores, fitted = {}, {}
for m in SIX:
    for k in CAL:
        f = fit_map(k, scores[m]["val"], y_val)
        cal_scores[(m, k)] = dict(val=f(scores[m]["val"]), test=f(scores[m]["test"]))
        if k == "temperature": fitted[(m, "T")] = TemperatureScaling().fit(scores[m]["val"], y_val).T_
        if k == "beta":        fitted[(m, "beta")] = BetaCalibration().fit(scores[m]["val"], y_val).params_.round(4).tolist()
np.savez_compressed(SCORES / f"scores_d1_seed{SEED}_calibrated.npz", y_val=y_val, y_te=y_te,
                    amount_val=amount_val, amount_te=amount_te,
                    **{f"{m}_{k}_{s}": cal_scores[(m, k)][s] for m in SIX for k in CAL for s in ("val", "test")})
for s, y, amt in (("val", y_val, amount_val), ("test", y_te, amount_te)):
    df = pd.DataFrame({"y": y, "amount": amt})
    for m in SIX:
        for k in CAL:
            df[f"{m}_{ {'none':'raw','temperature':'TS','beta':'Beta'}[k] }"] = cal_scores[(m, k)][s]
    df.to_csv(SCORES / f"scores_d1_seed{SEED}_{s}.csv", index=False, float_format="%.10g")
print("fitted temperatures:", {m: round(fitted[(m, 'T')], 4) for m in SIX})
print("fitted Beta (a, b, c):", {m: fitted[(m, 'beta')] for m in SIX})

# cross-check against the released module when it can be imported (needs the boosting libraries)
try:
    import c3e_contribution3 as c3
    for m in SIX:
        T_ref = c3.TemperatureScaling().fit(scores[m]["val"], y_val).T_
        assert abs(T_ref - fitted[(m, "T")]) < 1e-9, (m, T_ref, fitted[(m, "T")])
    print("c3e_contribution3.TemperatureScaling reproduces the inline copy exactly.")
except ImportError:
    print("c3e_contribution3 not importable here (boosting libraries missing) - inline copies used.")

fitted temperatures: {'LR': 0.6382, 'RF': 0.7487, 'XGB': 1.2894, 'LGBM': 4.8626, 'CatBoost': 0.1833, 'IF-Hybrid': 1.2702}
fitted Beta (a, b, c): {'LR': [0.7562, 0.6101, -6.8027], 'RF': [1.1464, 3.2579, -0.7705], 'XGB': [0.5948, 0.689, -1.8334], 'LGBM': [1.0, 0.0001, -3.7571], 'CatBoost': [1.2073, 5.5979, -9.4422], 'IF-Hybrid': [0.6296, 0.6412, -1.5866]}


c3e_contribution3.TemperatureScaling reproduces the inline copy exactly.


## 2 — Table 7: extended comparison on D1 from the saved scores

In [3]:
grid = np.linspace(0.0, 1.0, N_GRID)
def cost_at(y, p, tau, cfn=CFN, cfp=CFP):
    yp = p >= tau
    return float(cfn * np.sum((y == 1) & (~yp)) + cfp * np.sum((y == 0) & yp))
def cost_curve(y, p, g): return np.array([cost_at(y, p, t) for t in g], float)

rows = []
for m in SIX:
    pv = scores[m]["val"]
    row = {"model": m, "delta": round(float(roc_auc_score(y_val, pv) - average_precision_score(y_val, pv)), 3)}
    for k in CAL:
        a, b = cal_scores[(m, k)]["val"], cal_scores[(m, k)]["test"]
        tau = float(grid[int(np.argmin(cost_curve(y_val, a, grid)))])
        row[f"ECE_{k}"] = round(ece_10bins(y_val, np.clip(a, 0, 1)), 3)
        row[f"cost_{k}"] = round(cost_at(y_te, b, tau), 1)
        row[f"flag_{k}"] = int(np.sum(b >= tau))
        row[f"tau_{k}"] = round(tau, 4)
    rows.append(row)
tab7 = pd.DataFrame(rows)[["model", "delta", "ECE_none", "ECE_temperature", "ECE_beta",
                           "cost_none", "cost_temperature", "cost_beta",
                           "flag_none", "flag_temperature", "flag_beta", "tau_none", "tau_temperature", "tau_beta"]]
tab7.to_csv(RES / "tab7_extended_d1.csv", index=False)
print(tab7.to_string(index=False))

    model  delta  ECE_none  ECE_temperature  ECE_beta  cost_none  cost_temperature  cost_beta  flag_none  flag_temperature  flag_beta  tau_none  tau_temperature  tau_beta
       LR  0.202     0.082            0.053       0.0      202.0             216.0      221.0         79               115         76     0.999            0.999     0.087
       RF  0.197     0.001            0.000       0.0      192.0             192.0      192.0         80                80         80     0.139            0.080     0.072
      XGB  0.192     0.000            0.000       0.0      197.0             197.0      197.0         63                63         63     0.861            0.805     0.363
     LGBM  0.868     0.000            0.067       0.0     1858.0             750.0      750.0       1768                 0          0     1.000            0.965     0.023
 CatBoost  0.250     0.225            0.019       0.0      231.0             231.0      231.0         64                64         64     0.782  

## 3 — Table 8: threshold-grid audit on the saved scores

For each model and each grid (uniform 1001 points; every distinct validation score; 1001 empirical quantiles) we record the validation cost actually minimised, the number of tied minimisers, the frozen threshold, the number of test transactions it flags, the test cost, and the oracle (best non-empty rule chosen on the test scores themselves).

In [4]:
def three_grids(p_val):
    return {"uniform":  grid,
            "unique":   np.unique(p_val),
            "quantile": np.unique(np.quantile(p_val, np.linspace(0.0, 1.0, N_GRID)))}

def select_and_audit(p_val, p_te, g, label, model, calib):
    g = np.unique(np.clip(g, 0.0, 1.0))
    cv = cost_curve(y_val, p_val, g); cmin = cv.min()
    tied = np.flatnonzero(cv == cmin)
    tau = float(g[tied[0]])                                  # np.argmin convention: smallest tied threshold
    te_tied = np.array([cost_at(y_te, p_te, float(g[j])) for j in tied])
    oracle = float(cost_curve(y_te, p_te, np.unique(p_te)).min())   # non-empty rules only
    return dict(model=model, calibrator=calib, grid=label,
                n_val_unique=int(np.unique(p_val).size), tau_star=round(tau, 6),
                val_cost=round(cmin, 1), n_tied_val_optima=int(tied.size),
                n_flagged_test=int(np.sum(p_te >= tau)), test_cost=round(cost_at(y_te, p_te, tau), 1),
                test_cost_tied_min=round(float(te_tied.min()), 1), test_cost_tied_max=round(float(te_tied.max()), 1),
                test_cost_oracle=round(oracle, 1))

rows = []
for m in MODELS:
    pv, pt = scores[m]["val"], scores[m]["test"]
    d = roc_auc_score(y_val, pv) - average_precision_score(y_val, pv)
    for gname, g in three_grids(pv).items():
        r = select_and_audit(pv, pt, g, gname, m, "none"); r["delta"] = round(float(d), 3); rows.append(r)
pv_ts, pt_ts = cal_scores[("LGBM", "temperature")]["val"], cal_scores[("LGBM", "temperature")]["test"]
d_ts = roc_auc_score(y_val, pv_ts) - average_precision_score(y_val, pv_ts)
for gname, g in three_grids(pv_ts).items():
    r = select_and_audit(pv_ts, pt_ts, g, gname, "LGBM", "temperature"); r["delta"] = round(float(d_ts), 3); rows.append(r)
tab8 = pd.DataFrame(rows)[["model", "calibrator", "grid", "delta", "n_val_unique", "tau_star", "val_cost",
                           "n_tied_val_optima", "n_flagged_test", "test_cost", "test_cost_tied_min",
                           "test_cost_tied_max", "test_cost_oracle"]]
tab8.to_csv(RES / "tab8_gridres.csv", index=False)
print(tab8.to_string(index=False))
print(f"\nnull-model cost (test) = C_FN x N1_test = {CFN:.0f} x {int(y_te.sum())} = {NULL_COST_TEST:.0f}")
empty = tab8[tab8.n_flagged_test == 0]
print("rows whose frozen threshold flags no test transaction:\n", empty[["model", "calibrator", "grid", "tau_star", "test_cost"]].to_string(index=False))

model  calibrator     grid  delta  n_val_unique  tau_star  val_cost  n_tied_val_optima  n_flagged_test  test_cost  test_cost_tied_min  test_cost_tied_max  test_cost_oracle
   LR        none  uniform  0.202         55855  0.999000     155.0                  1              79      202.0               202.0               202.0             193.0
   LR        none   unique  0.202         55855  0.999559     154.0                  1              76      221.0               221.0               221.0             193.0
   LR        none quantile  0.202         55855  0.999917     165.0                  1              65      254.0               254.0               254.0             193.0
   RF        none  uniform  0.197           117  0.139000     142.0                  4              80      192.0               188.0               192.0             185.0
   RF        none   unique  0.197           117  0.140000     142.0                  2              80      192.0               188.0       

## 4 — Where the 750 comes from: ε sweep and the strictly increasing rank map (Section 6.2)

In [5]:
pv_raw, pt_raw = scores["LGBM"]["val"], scores["LGBM"]["test"]
def rank_map(p_ref):
    u = np.unique(p_ref); t = (np.arange(1, u.size + 1) - 0.5) / u.size
    return lambda s: np.interp(np.asarray(s, float), u, t, left=0.0, right=1.0)

rows = []
for e in EPS_LIST:
    tse = TemperatureScaling(eps=e).fit(pv_raw, y_val)
    a, b = tse.predict(pv_raw), tse.predict(pt_raw)
    r = select_and_audit(a, b, grid, "uniform", "LGBM", f"TS(eps={e:.0e})")
    r.update(eps=e, T=round(tse.T_, 4), n_out_unique=int(np.unique(a).size)); rows.append(r)
phi = rank_map(pv_raw); a, b = phi(pv_raw), phi(pt_raw)
r = select_and_audit(a, b, grid, "uniform", "LGBM", "rank map (strictly increasing)")
r.update(eps=np.nan, T=np.nan, n_out_unique=int(np.unique(a).size)); rows.append(r)
r = select_and_audit(pv_raw, pt_raw, grid, "uniform", "LGBM", "none")
r.update(eps=np.nan, T=np.nan, n_out_unique=int(np.unique(pv_raw).size)); rows.append(r)
eps_tab = pd.DataFrame(rows)[["calibrator", "eps", "T", "n_out_unique", "tau_star", "val_cost",
                              "n_tied_val_optima", "n_flagged_test", "test_cost", "test_cost_oracle"]]
eps_tab.to_csv(RES / "tab_eps_sensitivity.csv", index=False)
print(eps_tab.to_string(index=False))

# the clip audit quoted in Section 6.2
e = 1e-7
print(f"\nLGBM: {np.unique(pv_raw).size} distinct raw validation scores; {int((pv_raw <= e).sum()):,} at or below eps, "
      f"{int((pv_raw >= 1 - e).sum()):,} at or above 1-eps; {np.unique(pv_ts).size} distinct after TS "
      f"(T = {fitted[('LGBM','T')]:.3f}, range [{pv_ts.min():.4f}, {pv_ts.max():.4f}])")
print(f"largest raw validation score = {pv_raw.max():.4f}; largest mapped score = {pv_ts.max():.4f} < 0.965 = frozen tau")

                    calibrator          eps       T  n_out_unique  tau_star  val_cost  n_tied_val_optima  n_flagged_test  test_cost  test_cost_oracle
                 TS(eps=1e-06) 1.000000e-06  4.1680             3     0.965     570.0                 36               0      750.0            1858.0
                 TS(eps=1e-07) 1.000000e-07  4.8626             3     0.965     570.0                 36               0      750.0            1858.0
                 TS(eps=1e-09) 1.000000e-09  6.2517             3     0.965     570.0                 36               0      750.0            1858.0
                 TS(eps=1e-12) 1.000000e-12  8.3355             4     0.965     570.0                 36               0      750.0            1858.0
                 TS(eps=1e-15) 1.000000e-15 10.4189             4     0.965     570.0                 36               0      750.0            1858.0
rank map (strictly increasing)          NaN     NaN            21     0.977     570.0               

## 5 — Δ on raw versus mapped scores (Proposition 3.1 holds exactly only for strictly increasing maps)

In [6]:
variants = {"raw": pv_raw, "TS (eps=1e-7)": pv_ts, "rank map": phi(pv_raw), "Beta": cal_scores[("LGBM", "beta")]["val"]}
rows = [dict(variant=k, n_unique=int(np.unique(v).size),
             roc_auc=float(roc_auc_score(y_val, v)), pr_auc=float(average_precision_score(y_val, v))) for k, v in variants.items()]
dtab = pd.DataFrame(rows); dtab["delta"] = dtab.roc_auc - dtab.pr_auc
dtab.round(6).to_csv(RES / "tab_delta_after_calibration.csv", index=False)
print(dtab.round(6).to_string(index=False))
d_raw, d_map = dtab.set_index("variant").delta[["raw", "TS (eps=1e-7)"]]
print(f"\nDelta raw = {d_raw:.5f} ; Delta after TS = {d_map:.5f} ; difference = {d_map - d_raw:.1e} "
      f"(ties created by the clip; the rank map, which is strictly increasing, gives exactly {dtab.set_index('variant').delta['rank map']:.5f})")

      variant  n_unique  roc_auc   pr_auc    delta
          raw        21 0.886032 0.018471 0.867560
TS (eps=1e-7)         3 0.886057 0.018462 0.867594
     rank map        21 0.886032 0.018471 0.867560
         Beta         3 0.886057 0.018462 0.867594

Delta raw = 0.86756 ; Delta after TS = 0.86759 ; difference = 3.4e-05 (ties created by the clip; the rank map, which is strictly increasing, gives exactly 0.86756)


## 6 — Remark 3.3: the counter-example and the 2,000-draw simulation (seed 42)

Narrow-support score sets (width 0.05 around 0.47–0.52), a strictly increasing logistic map $\phi(p) = \sigma(k(p-\bar p))$ with $k \sim U(2, 12)$, and a coarse grid of spacing 0.1. The draw uses `np.random.default_rng(42)`; the fractions 6.0 % / 5.8 % quoted in the paper are those printed below.

In [7]:
def best_on(y, p, g):
    c = np.array([CFN * np.sum((p < t)[y == 1]) + CFP * np.sum((p >= t)[y == 0]) for t in g], float)
    return float(g[int(np.argmin(c))]), float(c.min())
def reachable_cuts(p, g): return len({tuple(np.asarray(p) >= t) for t in g})

y_ce = np.array([0, 0, 1]); p_ce = np.array([0.48, 0.49, 0.51]); q_ce = np.array([0.1, 0.6, 0.9])
g_ce = np.array([0.0, 0.5, 1.0])
t1, c1 = best_on(y_ce, p_ce, g_ce); t2, c2 = best_on(y_ce, q_ce, g_ce)
ce = pd.DataFrame([dict(scores="0.48 / 0.49 / 0.51", width=round(float(np.ptp(p_ce)), 2), tau=t1, cost=c1, reachable_rules=reachable_cuts(p_ce, g_ce)),
                   dict(scores="0.10 / 0.60 / 0.90", width=round(float(np.ptp(q_ce)), 2), tau=t2, cost=c2, reachable_rules=reachable_cuts(q_ce, g_ce))])
print(ce.to_string(index=False))

rng = np.random.default_rng(SEED)
n_worse = n_better = n_equal = 0; N_SIM = 2000
for _ in range(N_SIM):
    n = int(rng.integers(20, 60)); y = (rng.random(n) < 0.2).astype(int)
    p = np.sort(rng.random(n) * 0.05 + 0.47)
    k = float(rng.uniform(2.0, 12.0))
    q = 1.0 / (1.0 + np.exp(-k * (p - p.mean())))
    G = np.linspace(0, 1, 11)
    _, cp = best_on(y, p, G); _, cq = best_on(y, q, G)
    n_worse += cq > cp; n_better += cq < cp; n_equal += cq == cp
sim_worse, sim_better = 100 * n_worse / N_SIM, 100 * n_better / N_SIM
print(f"\nsimulation ({N_SIM} draws, grid spacing 0.1, seed {SEED}): validation cost degraded in {sim_worse:.1f} %, "
      f"improved in {sim_better:.1f} %, unchanged in {100 * n_equal / N_SIM:.1f} %")
ce.assign(sim_pct_worse=sim_worse, sim_pct_better=sim_better).to_csv(RES / "tab_counterexample.csv", index=False)

            scores  width  tau  cost  reachable_rules
0.48 / 0.49 / 0.51   0.03  0.5   0.0                3
0.10 / 0.60 / 0.90   0.80  0.5   1.0                3



simulation (2000 draws, grid spacing 0.1, seed 42): validation cost degraded in 6.0 %, improved in 5.8 %, unchanged in 88.2 %


## 7 — Table 10: the 26 pairs, their labels, and the κ sweep

A pair is *costly* when its five-seed mean test cost exceeds κ times the lowest mean cost on the same dataset (κ = 2 in the paper), *regular* otherwise. The DNN rows of the multi-seed CSV are not part of Table 5 and are excluded.

In [8]:
raw = pd.read_csv(RES / "c3e_all_seeds_raw.csv")
raw = raw[~raw["model"].isin(["DNN", "DNN-Platt", "NODE", "TabNet"])].copy()
pairs = (raw.groupby(["dataset", "model"], as_index=False)
            .agg(delta=("delta", "mean"), cost=("expected_cost", "mean"), tau_star=("tau_star", "mean"), n_seeds=("delta", "size")))
pairs["best_cost_ds"] = pairs.groupby("dataset")["cost"].transform("min")
pairs["cost_ratio"] = pairs["cost"] / pairs["best_cost_ds"]
pairs["case"] = np.where(pairs["cost_ratio"] > KAPPA_MAIN, "costly", "regular")   
pairs = pairs.sort_values(["dataset", "model"]).reset_index(drop=True)
pairs.to_csv(RES / "tab10_pairs_labelled.csv", index=False)
assert len(pairs) == 26 and (pairs.n_seeds == 5).all()
print(pairs[["dataset", "model", "delta", "cost", "cost_ratio", "case"]].to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

rows = []
for k in KAPPA_LIST:
    case = np.where(pairs["cost_ratio"] > k, "costly", "regular")
    for d in DELTA_STAR_LIST:
        flag = pairs["delta"].to_numpy() > d
        missed = pairs.loc[(case == "costly") & (~flag), ["dataset", "model"]]
        needless = pairs.loc[(case == "regular") & flag, ["dataset", "model"]]
        rows.append(dict(kappa=k, delta_star=d, n_flagged=int(flag.sum()), n_total=len(pairs),
                         n_costly=int((case == "costly").sum()), n_regular=int((case == "regular").sum()),
                         n_missed=len(missed), n_needless=len(needless),
                         missed="; ".join(f"{r.dataset}/{r.model}" for r in missed.itertuples()) or "-"))
scr = pd.DataFrame(rows); scr.to_csv(RES / "tab10_screening.csv", index=False)
print("\nTable 10 (kappa = 2):\n", scr[scr.kappa == KAPPA_MAIN].to_string(index=False))
print("\nsensitivity to kappa:\n", scr[["kappa", "delta_star", "n_flagged", "n_costly", "n_missed", "n_needless"]].to_string(index=False))

   dataset    model  delta        cost  cost_ratio    case
       baf     LGBM  0.735  24,083.800       1.008 regular
       baf       LR  0.748  24,590.000       1.029 regular
       baf       RF  0.727  24,023.200       1.006 regular
       baf      XGB  0.734  23,891.200       1.000 regular
creditcard     LGBM  0.868   1,858.000       9.718  costly
creditcard       LR  0.202     202.000       1.056 regular
creditcard       RF  0.186     195.600       1.023 regular
creditcard      XGB  0.187     191.200       1.000 regular
  elliptic CatBoost  0.029   2,258.600       1.011 regular
  elliptic     LGBM  0.029   2,234.000       1.000 regular
  elliptic       LR  0.403   3,483.000       1.559 regular
  elliptic       RF  0.033   2,267.600       1.015 regular
  elliptic      XGB  0.029   2,261.000       1.012 regular
    giveme CatBoost  0.455  10,187.000       1.000 regular
    giveme     LGBM  0.458  10,345.000       1.016 regular
    giveme       LR  0.451  11,331.000       1.112 regul

## 8 — Table 9 (fixed vs logarithmic regime) and the three regimes of Fig. 11

Fixed: $C_{FN}=10$. Linear: $C_{FN}^{(i)}=\max(\mathrm{Amount}_i,1)$. Logarithmic: $C_{FN}^{(i)}=\log(1+\mathrm{Amount}_i)$. $C_{FP}=1$ throughout; the threshold is selected on validation under each regime and frozen.

In [9]:
W = {"fixed":  (np.full(len(y_val), CFN), np.full(len(y_te), CFN)),
     "linear": (np.maximum(amount_val, 1.0), np.maximum(amount_te, 1.0)),
     "log":    (np.log1p(amount_val), np.log1p(amount_te))}
def cost_w(y, p, tau, w):
    yp = p >= tau
    return float(np.sum(w[(y == 1) & (~yp)]) + CFP * np.sum((y == 0) & yp))
rows9, rows11 = [], []
for m in SIX:
    pv, pt = scores[m]["val"], scores[m]["test"]
    r9, r11 = {"model": m}, {"model": m}
    for reg, (wv, wt) in W.items():
        tau = float(grid[int(np.argmin([cost_w(y_val, pv, t, wv) for t in grid]))])
        fn = int(np.sum((y_te == 1) & (pt < tau))); fp = int(np.sum((y_te == 0) & (pt >= tau)))
        r11[f"tau_{reg}"], r11[f"cost_{reg}"] = round(tau, 4), round(cost_w(y_te, pt, tau, wt), 2)
        if reg in ("fixed", "log"):
            r9[f"tau_{reg}"], r9[f"FN_{reg}"], r9[f"FP_{reg}"], r9[f"cost_{reg}"] = round(tau, 4), fn, fp, round(cost_w(y_te, pt, tau, wt), 2)
    rows9.append(r9); rows11.append(r11)
tab9 = pd.DataFrame(rows9)[["model", "tau_fixed", "FN_fixed", "FP_fixed", "cost_fixed", "tau_log", "FN_log", "FP_log", "cost_log"]]
tab9.to_csv(RES / "tab9_costregimes.csv", index=False)
tab11 = pd.DataFrame(rows11).set_index("model"); tab11.to_csv(RES / "fig11_cost_regimes_d1.csv")
print(tab9.to_string(index=False)); print(); print(tab11.to_string())
unit_log = float(np.log1p(amount_te)[y_te == 1].mean())
print(f"\nmean C_FN per test fraud: fixed {CFN:.0f} | log {unit_log:.2f}  -> unit ratio {CFN / unit_log:.2f}")
print("ratio fixed/log per model:", (tab9.set_index('model').cost_fixed / tab9.set_index('model').cost_log).round(2).to_dict())

    model  tau_fixed  FN_fixed  FP_fixed  cost_fixed  tau_log  FN_log  FP_log  cost_log
       LR      0.999        18        22       202.0    0.999      18      22     76.02
       RF      0.139        17        22       192.0    0.367      20       7     65.30
      XGB      0.861        19         7       197.0    0.861      19       7     67.93
     LGBM      1.000        15      1708      1858.0    1.000      15    1708   1756.39
 CatBoost      0.782        22        11       231.0    0.806      24       3     64.54
IF-Hybrid      0.096        18        21       201.0    0.822      18       7     61.02

           tau_fixed  cost_fixed  tau_linear  cost_linear  tau_log  cost_log
model                                                                       
LR             0.999       202.0       0.884      2595.28    0.999     76.02
RF             0.139       192.0       0.013      2789.03    0.367     65.30
XGB            0.861       197.0       0.001      2070.79    0.861     67.9

## 9 — Assertions: the numbers quoted in the paper

In [10]:
t8 = tab8.set_index(["model", "calibrator", "grid"])
lg_ts = t8.loc[("LGBM", "temperature", "uniform")]; lg_raw = t8.loc[("LGBM", "none", "uniform")]
checks = {
    "Table 8: LGBM raw uniform test cost 1858":              lg_raw.test_cost == 1858,
    "Table 8: LGBM raw 21 distinct validation scores":       lg_raw.n_val_unique == 21,
    "Table 8: LGBM+TS uniform flags 0 and costs 750":        (lg_ts.n_flagged_test == 0) and (lg_ts.test_cost == NULL_COST_TEST == 750),
    "Table 8: LGBM+TS uniform tau 0.965, 36 ties, val cost 570": (lg_ts.tau_star == 0.965) and (lg_ts.n_tied_val_optima == 36) and (lg_ts.val_cost == 570),
    "Table 8: LGBM+TS unique / quantile grids return 1858":  (t8.loc[("LGBM","temperature","unique")].test_cost == 1858) and (t8.loc[("LGBM","temperature","quantile")].test_cost == 1858),
    "Table 8: oracle for LGBM is 1858 (>= null model 750)":  lg_raw.test_cost_oracle == 1858,
    "Table 8: LR quantile 254 vs uniform 202":               (t8.loc[("LR","none","quantile")].test_cost == 254) and (t8.loc[("LR","none","uniform")].test_cost == 202),
    "eps sweep: test cost 750 for every eps and for the rank map": (eps_tab[eps_tab.calibrator != "none"].test_cost == 750).all(),
    "eps sweep: 3 distinct TS scores at eps=1e-7, range max 0.9649": (eps_tab.set_index("calibrator").loc["TS(eps=1e-07)"].n_out_unique == 3) and (round(float(pv_ts.max()), 4) == 0.9649),
    "Section 6.2: fitted T = 4.863; 54,928 at clip-low; 2,032 at clip-high":
        (round(fitted[("LGBM","T")], 3) == 4.863) and (int((pv_raw <= 1e-7).sum()) == 54928) and (int((pv_raw >= 1 - 1e-7).sum()) == 2032),
    "Section 6.2: raw flags 1,768 test transactions":         lg_raw.n_flagged_test == 1768,
    "Section 6.2: Delta 0.868 raw and mapped; AUCs 0.8860/0.0185 vs 0.8861/0.0185":
        (round(d_raw, 3) == 0.868) and (round(d_map, 3) == 0.868)
        and (round(float(roc_auc_score(y_val, pv_raw)), 4) == 0.8860) and (round(float(roc_auc_score(y_val, pv_ts)), 4) == 0.8861)
        and (round(float(average_precision_score(y_val, pv_raw)), 4) == 0.0185) and (round(float(average_precision_score(y_val, pv_ts)), 4) == 0.0185),
    "Remark 3.3: counter-example cost 0 -> 1":                (c1 == 0) and (c2 == 1),
    "Remark 3.3: simulation 6.0 % worse / 5.8 % better":      (round(sim_worse, 1) == 6.0) and (round(sim_better, 1) == 5.8),
    "Table 7: RF 192/192/192, XGB 197/197/197, CatBoost 231, IF-Hybrid 201/201/200, LR 202/216/221, LGBM 1858/750/750":
        tab7.set_index("model")[["cost_none","cost_temperature","cost_beta"]].astype(int).values.tolist()
        == [[202,216,221],[192,192,192],[197,197,197],[1858,750,750],[231,231,231],[201,201,200]],
    "Table 7: Delta LR .202 RF .197 XGB .192 LGBM .868 CatBoost .250 IF .193":
        tab7.delta.tolist() == [0.202, 0.197, 0.192, 0.868, 0.250, 0.193],
    "Table 7: ECE raw CatBoost 0.225, LGBM TS 0.067, LR 0.082 -> 0.053 -> 0.000":
        (tab7.set_index("model").loc["CatBoost","ECE_none"] == 0.225) and (tab7.set_index("model").loc["LGBM","ECE_temperature"] == 0.067)
        and tab7.set_index("model").loc["LR",["ECE_none","ECE_temperature","ECE_beta"]].tolist() == [0.082, 0.053, 0.0],
    "Table 10: 26 pairs, 2 costly / 24 regular at kappa=2; flagged 22/22/20/19/19; missed 0; needless 20/20/18/17/17":
        (len(pairs) == 26) and scr[scr.kappa == 2.0].n_costly.tolist() == [2]*5
        and scr[scr.kappa == 2.0].n_flagged.tolist() == [22,22,20,19,19] and scr[scr.kappa == 2.0].n_missed.tolist() == [0]*5
        and scr[scr.kappa == 2.0].n_needless.tolist() == [20,20,18,17,17],
    "Table 10: PaySim/LGBM Delta 0.635 (five-seed mean 0.6345) is flagged at 0.30":
        round(float(pairs.set_index(["dataset","model"]).loc[("paysim","LGBM"),"delta"]), 3) == 0.635 and float(pairs.set_index(["dataset","model"]).loc[("paysim","LGBM"),"delta"]) > 0.30,
    "Table 9: log costs 76.02 / 65.30 / 67.93 / 1756.39 / 64.54 / 61.02; unit ratio 4.12":
        tab9.cost_log.tolist() == [76.02, 65.30, 67.93, 1756.39, 64.54, 61.02] and round(CFN / unit_log, 2) == 4.12,
    "Table 9: LR, XGB, LGBM keep the same operating point; IF-Hybrid moves 0.096 -> 0.822":
        all(tab9.set_index("model").loc[m, "tau_fixed"] == tab9.set_index("model").loc[m, "tau_log"] for m in ["LR","XGB","LGBM"])
        and tab9.set_index("model").loc["IF-Hybrid", ["tau_fixed","tau_log"]].tolist() == [0.096, 0.822],
    "Fig. 11: linear > fixed > log for all six configurations": bool(((tab11.cost_linear > tab11.cost_fixed) & (tab11.cost_fixed > tab11.cost_log)).all()),
}
w = max(len(k) for k in checks)
for k, ok in checks.items(): print(f"[{'OK  ' if ok else 'FAIL'}] {k}")
assert all(bool(v) for v in checks.values()), "a quoted number does not regenerate - fix the paper, not the notebook"
print("\nAll quoted numbers regenerate from the saved scores.")

[OK  ] Table 8: LGBM raw uniform test cost 1858
[OK  ] Table 8: LGBM raw 21 distinct validation scores
[OK  ] Table 8: LGBM+TS uniform flags 0 and costs 750
[OK  ] Table 8: LGBM+TS uniform tau 0.965, 36 ties, val cost 570
[OK  ] Table 8: LGBM+TS unique / quantile grids return 1858
[OK  ] Table 8: oracle for LGBM is 1858 (>= null model 750)
[OK  ] Table 8: LR quantile 254 vs uniform 202
[OK  ] eps sweep: test cost 750 for every eps and for the rank map
[OK  ] eps sweep: 3 distinct TS scores at eps=1e-7, range max 0.9649
[OK  ] Section 6.2: fitted T = 4.863; 54,928 at clip-low; 2,032 at clip-high
[OK  ] Section 6.2: raw flags 1,768 test transactions
[OK  ] Section 6.2: Delta 0.868 raw and mapped; AUCs 0.8860/0.0185 vs 0.8861/0.0185
[OK  ] Remark 3.3: counter-example cost 0 -> 1
[OK  ] Remark 3.3: simulation 6.0 % worse / 5.8 % better
[OK  ] Table 7: RF 192/192/192, XGB 197/197/197, CatBoost 231, IF-Hybrid 201/201/200, LR 202/216/221, LGBM 1858/750/750
[OK  ] Table 7: Delta LR .202 RF .19